# 05 Financial KPI Calculations
**Project:** Retailmart Finance Analytics  
**Phase:** Compute key financial metrics (Gross Revenue, Net Revenue, COGS, Profit Margins).

---

### 1. Setup & Imports
We must add the parent directory to the system path so Python can locate the custom `src` modules.

In [15]:
import sys
import os
import pandas as pd
import numpy as np

# Add root directory to python path
sys.path.append(os.path.abspath(os.path.join('..')))

# Import custom configurations and utilities
from src.config.config import config
from src.database.connection import db_connector
from src.utils.logger import setup_logger
from src.utils.helpers import save_dataframe

from src.utils.helpers import format_large_number

logger = setup_logger(os.path.splitext(os.path.basename('__file__'))[0])
logger.info('Notebook setup and imports loaded successfully.')

[2026-07-19 16:15:42] [INFO] [1329643734.py:18] Notebook setup and imports loaded successfully.


### 2. Load Cleaned Data
Placeholder for load cleaned data steps.

In [16]:
import os

# 1. Define paths to our processed clean files
processed_dir = "../data/processed"

# 2. Load the orders table, telling Pandas to parse 'order_date' as a real date
orders_df = pd.read_csv(
    os.path.join(processed_dir, "sales_transaction_orders.csv"),
    parse_dates=["order_date"]
)

# 3. Load the general ledger entries, parsing transaction dates
ledger_df = pd.read_csv(
    os.path.join(processed_dir, "finance_accounting_ledger_entries.csv"),
    parse_dates=["transaction_date", "posted_at"]
)

# 4. Load lookup tables (no date parsing needed for these)
gl_accounts_df = pd.read_csv(os.path.join(processed_dir, "finance_accounting_gl_accounts.csv"))
dealers_df = pd.read_csv(os.path.join(processed_dir, "dealer_dealers.csv"))

# 5. Print shapes to confirm they loaded successfully
print(f"Loaded Orders: {orders_df.shape}")
print(f"Loaded Ledger Entries: {ledger_df.shape}")
print(f"Loaded GL Accounts: {gl_accounts_df.shape}")
print(f"Loaded Dealers: {dealers_df.shape}")

Loaded Orders: (7000, 16)
Loaded Ledger Entries: (15000, 10)
Loaded GL Accounts: (80, 6)
Loaded Dealers: (200, 16)


### 3. Compute Monthly Revenue & Costs
Placeholder for compute monthly revenue & costs steps.

In [17]:
# 1. Create a Year-Month column for our orders
orders_df["order_month"] = orders_df["order_date"].dt.to_period("M")

# 2. Calculate Monthly Vehicle Revenue from the orders table
monthly_vehicle_revenue = orders_df.groupby("order_month")["net_amount"].sum().reset_index()
monthly_vehicle_revenue.rename(columns={"net_amount": "vehicle_revenue"}, inplace=True)

# 3. Join the Ledger Entries with GL Accounts to get account names/codes
merged_ledger = pd.merge(ledger_df, gl_accounts_df, on="account_id")

# 4. Create a Year-Month column for our ledger entries
merged_ledger["transaction_month"] = merged_ledger["transaction_date"].dt.to_period("M")

# 5. Extract category codes (the first 4 digits of account_code)
# e.g., '5000-029' becomes '5000' (Cost of Goods Sold category)
merged_ledger["category_code"] = merged_ledger["account_code"].str[:4]

# 6. Separate Expenses (Debit) and other Revenues (Credit)
expenses_df = merged_ledger[merged_ledger["account_type"] == "Expense"]
other_income_df = merged_ledger[merged_ledger["account_type"] == "Income"]

# 7. Aggregate Monthly Expenses by Category
monthly_expenses = expenses_df.groupby(["transaction_month", "category_code"])["debit_amount"].sum().unstack(fill_value=0).reset_index()

# 8. Aggregate Monthly Other Revenues (Spares & Service Revenue)
monthly_other_income = other_income_df.groupby(["transaction_month", "category_code"])["credit_amount"].sum().unstack(fill_value=0).reset_index()

# 9. Let's look at the columns we just created to confirm
print("Monthly Vehicle Revenue Sample:")
print(monthly_vehicle_revenue.head(3))
print("\nMonthly Expenses Categories Sample:")
print(monthly_expenses.head(3))

Monthly Vehicle Revenue Sample:
  order_month  vehicle_revenue
0     2023-01      81585143.91
1     2023-02      56950148.59
2     2023-03      76857820.03

Monthly Expenses Categories Sample:
category_code transaction_month         5000         5100         5200  \
0                       2023-04   7537430.91   6965070.39   2588839.22   
1                       2023-05  38853936.16  37962864.74  18158302.15   
2                       2023-06  33211904.24  52192753.87  25111609.71   

category_code         5300         5400         5500         5600  
0               7425115.69    428081.98  12026056.88         0.00  
1              11982686.22  20060675.60  42880239.22  20367618.98  
2              16696606.00   6579100.23  80105959.05   4966456.90  


In [18]:
# --- Python equivalent of Phase 5 Challenge 1 ---
# 1. Create the Month Period column
orders_df["order_month"] = orders_df["order_date"].dt.to_period("M")

# 2. Group by month and calculate sum
monthly_vehicle_revenue = orders_df.groupby("order_month")["net_amount"].sum().reset_index()
monthly_vehicle_revenue.rename(columns={"net_amount": "vehicle_revenue"}, inplace=True)

# 3. Print the top 5 months to verify it matches pgAdmin
print(monthly_vehicle_revenue.head(5).to_string(index=False))

order_month  vehicle_revenue
    2023-01      81585143.91
    2023-02      56950148.59
    2023-03      76857820.03
    2023-04      84060290.72
    2023-05     328891440.45


### 4. Calculate Gross and Net Profit Margins
Placeholder for calculate gross and net profit margins steps.

In [19]:
# 1. Standardize month column names before merging
monthly_vehicle_revenue.rename(columns={"order_month": "month"}, inplace=True)
monthly_expenses.rename(columns={"transaction_month": "month"}, inplace=True)
monthly_other_income.rename(columns={"transaction_month": "month"}, inplace=True)

# 2. Merge all monthly sheets together on the common 'month' column
# We do an 'outer' merge to ensure we don't drop months if one table has missing entries
finance_master = pd.merge(monthly_vehicle_revenue, monthly_other_income, on="month", how="outer")
finance_master = pd.merge(finance_master, monthly_expenses, on="month", how="outer")

# 3. Fill any empty/NaN values with 0.0 (in case a month had zero transactions in a category)
finance_master.fillna(0.0, inplace=True)

# 4. Rename category codes to readable business names
rename_map = {
    "4100": "spares_revenue",
    "4200": "service_revenue",
    "5000": "cogs",
    "5100": "manufacturing_costs",
    "5200": "salaries_wages",
    "5300": "marketing_costs",
    "5400": "logistics_costs",
    "5500": "dealer_commissions",
    "5600": "admin_costs"
}
finance_master.rename(columns=rename_map, inplace=True)

# 5. Write the mathematical formulas for our KPIs
# Total Revenue = Vehicles + Spares + Services
finance_master["total_revenue"] = (
    finance_master["vehicle_revenue"] + 
    finance_master["spares_revenue"] + 
    finance_master["service_revenue"]
)

# Total Operating Expenses = Sum of all 7 cost columns
finance_master["total_expenses"] = (
    finance_master["cogs"] +
    finance_master["manufacturing_costs"] +
    finance_master["salaries_wages"] +
    finance_master["marketing_costs"] +
    finance_master["logistics_costs"] +
    finance_master["dealer_commissions"] +
    finance_master["admin_costs"]
)

# Gross Profit = Total Revenue - COGS
finance_master["gross_profit"] = finance_master["total_revenue"] - finance_master["cogs"]

# Net Profit = Total Revenue - Total Expenses
finance_master["net_profit"] = finance_master["total_revenue"] - finance_master["total_expenses"]

# Net Profit Margin (%) = (Net Profit / Total Revenue) * 100
finance_master["net_profit_margin"] = (finance_master["net_profit"] / finance_master["total_revenue"]) * 100

# 6. Display a sample of our final master sheet
print("--- Master Financial Summary Sheet ---")
display_cols = ["month", "total_revenue", "total_expenses", "gross_profit", "net_profit", "net_profit_margin"]
print(finance_master[display_cols].head(5).to_string(index=False))

--- Master Financial Summary Sheet ---
  month  total_revenue  total_expenses  gross_profit   net_profit  net_profit_margin
2023-01    81585143.91            0.00   81585143.91  81585143.91         100.000000
2023-02    56950148.59            0.00   56950148.59  56950148.59         100.000000
2023-03    76857820.03            0.00   76857820.03  76857820.03         100.000000
2023-04    91400265.70     36970595.07   83862834.79  54429670.63          59.550889
2023-05   400484763.85    190266323.07  361630827.69 210218440.78          52.490996


In [20]:
# --- Python equivalent of Phase 5 Challenge 2 ---
# 1. Join ledger entries with GL accounts
merged_ledger = pd.merge(ledger_df, gl_accounts_df, on="account_id")

# 2. Extract transaction month and category codes
merged_ledger["transaction_month"] = merged_ledger["transaction_date"].dt.to_period("M")
merged_ledger["category_code"] = merged_ledger["account_code"].str[:4]

# 3. Filter for Expenses
expenses_df = merged_ledger[merged_ledger["account_type"] == "Expense"]

# 4. Group by Month and Category, sum, and pivot columns
monthly_expenses = expenses_df.groupby(["transaction_month", "category_code"])["debit_amount"].sum().unstack(fill_value=0).reset_index()

# 5. Display the first 5 months of expenses
print(monthly_expenses.head(5).to_string(index=False))

transaction_month        5000        5100        5200        5300        5400        5500        5600
          2023-04  7537430.91  6965070.39  2588839.22  7425115.69   428081.98 12026056.88        0.00
          2023-05 38853936.16 37962864.74 18158302.15 11982686.22 20060675.60 42880239.22 20367618.98
          2023-06 33211904.24 52192753.87 25111609.71 16696606.00  6579100.23 80105959.05  4966456.90
          2023-07 45566351.48 53966949.65 15993431.42 14521635.80 11329824.47 47892824.46  4942715.27
          2023-08 45526534.92 30745127.74 25237166.94  4325450.97 17137836.24 99069796.16  5088378.79


### 5. Group by Region and Stores
Placeholder for group by region and stores steps.

In [22]:
# --- Python equivalent of Phase 5 Challenge 3 ---

# Step 0: Load the commissions table from our processed folder
processed_dir = "../data/processed"
commissions_df = pd.read_csv(os.path.join(processed_dir, "dealer_network_dealer_commissions.csv"))

# 1. Group orders by dealer_id to get orders count and sales sum
dealer_sales = orders_df.groupby("dealer_id").agg(
    total_orders=("order_id", "count"),
    total_revenue=("net_amount", "sum")
).reset_index()

# 2. Group commissions by dealer_id and sum
dealer_commissions = commissions_df.groupby("dealer_id")["total_commission"].sum().reset_index()

# 3. Merge sales, dealer metadata, and commissions (LEFT JOINS)
dealer_summary = pd.merge(dealer_sales, dealers_df[["dealer_id", "dealer_name"]], on="dealer_id", how="left")
dealer_summary = pd.merge(dealer_summary, dealer_commissions, on="dealer_id", how="left")

# 4. Handle Nulls (COALESCE)
dealer_summary["total_commission"] = dealer_summary["total_commission"].fillna(0.0)

# 5. Calculate payout efficiency percentage
dealer_summary["payout_ratio_pct"] = ((dealer_summary["total_commission"] / dealer_summary["total_revenue"]) * 100).round(2)

# 6. Sort and display top 5 showrooms
dealer_summary = dealer_summary.sort_values(by="total_revenue", ascending=False)
print(dealer_summary[["dealer_name", "total_orders", "total_revenue", "total_commission", "payout_ratio_pct"]].head(5).to_string(index=False))

            dealer_name  total_orders  total_revenue  total_commission  payout_ratio_pct
    Dada Motors Dhanbad            47    94050873.16        31049213.0             33.01
  Kar Motors Chandrapur            50    92021301.79        22320709.0             24.26
  Ahuja Motors Suryapet            45    89153735.34        13713162.0             15.38
Chaudhry Motors Bikaner            49    86400299.95        19440004.0             22.50
   Nagi Motors Durgapur            48    85810655.72        24923385.0             29.04


### 6. Export Finance Aggregates
Placeholder for export finance aggregates steps.

In [23]:
# --- Python equivalent of Phase 5 Challenge 4 ---
# 1. Standardize column names
monthly_vehicle_revenue.rename(columns={"order_month": "month"}, inplace=True)
monthly_expenses.rename(columns={"transaction_month": "month"}, inplace=True)
monthly_other_income.rename(columns={"transaction_month": "month"}, inplace=True)

# 2. Merge all monthly sheets together on the common 'month' column
finance_master = pd.merge(monthly_vehicle_revenue, monthly_other_income, on="month", how="outer")
finance_master = pd.merge(finance_master, monthly_expenses, on="month", how="outer")
finance_master.fillna(0.0, inplace=True)

# 3. Rename category codes to readable business names
rename_map = {
    "4100": "spares_revenue",
    "4200": "service_revenue",
    "5000": "cogs",
    "5100": "manufacturing_costs",
    "5200": "salaries_wages",
    "5300": "marketing_costs",
    "5400": "logistics_costs",
    "5500": "dealer_commissions",
    "5600": "admin_costs"
}
finance_master.rename(columns=rename_map, inplace=True)

# 4. Write the P&L math formulas
finance_master["total_revenue"] = (
    finance_master["vehicle_revenue"] + 
    finance_master["spares_revenue"] + 
    finance_master["service_revenue"]
)
finance_master["total_expenses"] = (
    finance_master["cogs"] +
    finance_master["manufacturing_costs"] +
    finance_master["salaries_wages"] +
    finance_master["marketing_costs"] +
    finance_master["logistics_costs"] +
    finance_master["dealer_commissions"] +
    finance_master["admin_costs"]
)
finance_master["gross_profit"] = finance_master["total_revenue"] - finance_master["cogs"]
finance_master["net_profit"] = finance_master["total_revenue"] - finance_master["total_expenses"]
finance_master["net_profit_margin"] = (finance_master["net_profit"] / finance_master["total_revenue"]) * 100

# 5. Display a sample of our final master sheet
print("--- Master Financial Summary Sheet ---")
display_cols = ["month", "total_revenue", "total_expenses", "gross_profit", "net_profit", "net_profit_margin"]
print(finance_master[display_cols].head(6).to_string(index=False))

--- Master Financial Summary Sheet ---
  month  total_revenue  total_expenses  gross_profit   net_profit  net_profit_margin
2023-01    81585143.91            0.00   81585143.91  81585143.91         100.000000
2023-02    56950148.59            0.00   56950148.59  56950148.59         100.000000
2023-03    76857820.03            0.00   76857820.03  76857820.03         100.000000
2023-04    91400265.70     36970595.07   83862834.79  54429670.63          59.550889
2023-05   400484763.85    190266323.07  361630827.69 210218440.78          52.490996
2023-06   340230307.43    218864390.00  307018403.19 121365917.43          35.671695
